In [0]:
dbutils.widgets.text("catalog", "claudecatalog", "Catálogo")
dbutils.widgets.text("schema", "supply_chain", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

source_path = f"/Volumes/{catalog}/{schema}/{volume}/incoming/"
checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoints/bronze_orders/"
schema_location = f"/Volumes/{catalog}/{schema}/{volume}/_schemas/bronze_orders/"

bronze_table = f"{catalog}.{schema}.bronze_orders"

print(f"Fuente: {source_path}")
print(f"Checkpoint: {checkpoint_path}")

Esto convierte, por ejemplo:

"Days for shipping (real)" → Days_for_shipping_real
"Benefit per order" → Benefit_per_order
"Customer Zipcode" → Customer_Zipcode

Importante: esto es una decisión de transformación, así que técnicamente ya no es "Bronze puro" en el sentido más estricto (Bronze idealmente es 1:1 con la fuente). Pero es una excepción aceptada en la industria: renombrar columnas por restricciones técnicas de almacenamiento (no por lógica de negocio) se hace en Bronze porque si no, ni siquiera puedes persistir los datos. Documenta esto en tu README como una decisión consciente — es exactamente el tipo de criterio que se evalúa en una entrevista técnica.

In [0]:
import re
from pyspark.sql.functions import current_timestamp, col

def sanitize_column_name(col_name):
    clean = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name)
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean

df_stream_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("header", "true")
    .option("encoding", "ISO-8859-1")
    .load(source_path)
)

new_column_names = [sanitize_column_name(c) for c in df_stream_raw.columns]
df_stream_raw = df_stream_raw.toDF(*new_column_names)

df_stream_bronze = (
    df_stream_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
query = (
    df_stream_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()
print(f"Stream completado. Tabla: {bronze_table}")
display(spark.table(bronze_table))
print(f"Total de filas: {spark.table(bronze_table).count()}")